# 02 · Strategic EDA — from charts to business narrative
*Section 6–7. Every chart answers an executive question and ends with a **so-what**.*

In [1]:
import sys, warnings; sys.path.insert(0, "..")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, plotly.express as px, plotly.graph_objects as go
from src import cleaning as cl, features as ft, segmentation as sg, viz
pd.options.display.float_format = "{:,.2f}".format
pd.options.display.max_columns = 40
df = pd.read_parquet("../data/processed/lines_clean.parquet")
orders = pd.read_parquet("../data/processed/orders.parquet")
sales = df[df.IsProductSale]
print(f"Net revenue (excl. accounting adjustments & zero-price): £{df[~df.IsAdjustment & ~df.IsZeroPrice].TotalLineRevenue.sum():,.0f}")
print(f"Gross product sales: £{sales.TotalLineRevenue.sum():,.0f}   |   orders: {len(orders):,}   |   customers: {sales.CustomerID.nunique():,}")

Net revenue (excl. accounting adjustments & zero-price): £9,737,069
Gross product sales: £10,247,905   |   orders: 19,773   |   customers: 4,334


## Time · how does revenue change, and when is the shop strongest?

In [2]:
viz.monthly_revenue(df)

**So-what.** Revenue is flat at roughly £0.5–0.75M/month until August, then **climbs Sep → Oct → Nov (£1.02M → £1.07M → £1.46M)**. November is ~2.2× the Mar–Aug average. December 2011 looks low only because the data stops on the 9th. Gift retailers buy stock early: the surge starts in *September*, so inventory and ad budgets must be committed by August.

In [3]:
viz.hour_weekday_heatmap(orders)

In [4]:
viz.weekday_bar(orders)

**So-what.** Trading happens **10:00–15:00** (peak at 12:00), ~84% of order value is booked between 09:00 and 16:00. **There is no Saturday trading at all**, and Sunday is the weakest day, Thursday the strongest (£2.13M). Scheduled maintenance belongs in the evening or on Saturday; paid-media bids should be raised for Tue/Thu late morning — the B2B buyer's working hours, which is itself evidence of trade customers.

## Products · what sells, and how concentrated is it?

In [5]:
viz.top_products(df, 'Revenue')

In [6]:
viz.top_products(df, 'Units')

In [7]:
p = viz.pareto(df); print(p.layout.meta); p

{'n80': 841, 'pct80': 21.547527542915706, 'n_products': 3903}


**So-what.** Net of returns the revenue leader is the **Regency Cakestand 3 Tier (£164k)**, while the unit leaders are cheap items (World War 2 Gliders, Jumbo Bags, Popcorn Holders). **The Pareto principle holds almost exactly: ~21.5% of products (≈840 of 3,900) produce 80% of net revenue.** The long tail of ~3,000 SKUs adds complexity and stock cost for 20% of sales.

In [8]:
dead = ft.dead_stock(df)
print(f"{len(dead)} products were in the top quartile of historical volume but sold ZERO or negative units in the last 90 days")
dead.head(12)

22 products were in the top quartile of historical volume but sold ZERO or negative units in the last 90 days


,Description,HistUnits,HistRevenue,LastSold,RecentNetUnits
StockCode,,,,,
47556B,TEA TIME TEA TOWELS,2600,"6,045.00",2011-04-18 13:20:00,0.00
22088,PAPER BUNTING COLOURED LACE,1837,"5,511.85",2011-07-06 10:41:00,0.00
21033,JUMBO BAG CHARLIE AND LOLA TOYS,1931,"4,849.94",2011-09-08 13:53:00,0.00
21658,GLASS BEURRE DISH,849,"3,596.49",2011-09-05 14:29:00,-3.00
85132C,CHARLIE AND LOLA FIGURES TINS,1515,"3,552.57",2011-09-02 09:38:00,0.00
22855,FINE WICKER HEART,2630,"3,286.73",2011-08-30 16:43:00,0.00
21870,I CAN ONLY PLEASE ONE PERSON MUG,1381,"1,818.96",2011-08-24 09:36:00,0.00
22198,LARGE POPCORN HOLDER,982,"1,801.56",2011-05-01 11:58:00,0.00
85206A,CREAM FELT EASTER EGG BASKET,974,"1,568.93",2011-03-17 18:24:00,0.00


**So-what.** These 22 former volume sellers (e.g. *Tea Time Tea Towels*, *Paper Bunting Coloured Lace*, the *Charlie & Lola* range) stopped selling abruptly — likely **delisted or stocked-out** rather than unpopular. Ops should check whether they were deliberately discontinued; if not, it is lost revenue.

## Customers & geography

In [9]:
c = pd.read_parquet("../data/processed/customers.parquet")
top = c.sort_values("Monetary", ascending=False)
share = lambda n: top.head(n).Monetary.sum() / c.Monetary.sum()
print(f"Top 10 customers = {share(10):.1%} of net revenue | top 20% of customers = {share(int(.2*len(c))):.1%}")
top[["CustomerID","Country","Frequency","Monetary","AvgBasketUnits"]].head(8)

Top 10 customers = 16.6% of net revenue | top 20% of customers = 73.7%


,CustomerID,Country,Frequency,Monetary,AvgBasketUnits
1689,14646,Netherlands,72,"278,778.02","2,733.94"
4197,18102,United Kingdom,60,"259,657.30","1,068.73"
3725,17450,United Kingdom,46,"189,575.53","1,521.15"
1879,14911,EIRE,198,"128,768.24",404.82
55,12415,Australia,20,"123,638.18","3,868.65"
1333,14156,EIRE,54,"113,685.77","1,069.54"
3768,17511,United Kingdom,31,"88,138.20","2,082.23"
3174,16684,United Kingdom,28,"65,920.12","1,794.82"


In [10]:
viz.country_chart(df)

In [11]:
g = sales.groupby("Country").agg(Revenue=("TotalLineRevenue","sum"), Orders=("InvoiceNo","nunique"), Customers=("CustomerID","nunique")).sort_values("Revenue", ascending=False)
g["Rev_per_customer"] = g.Revenue / g.Customers
print(f"UK share of product revenue: {g.loc['United Kingdom','Revenue']/g.Revenue.sum():.1%}")
g.head(8)

UK share of product revenue: 85.1%


,Revenue,Orders,Customers,Rev_per_customer
Country,,,,
United Kingdom,"8,725,129.48",17901,3916,"2,228.07"
Netherlands,"283,889.34",93,9,"31,543.26"
EIRE,"270,850.86",282,3,"90,283.62"
Germany,"205,381.15",443,94,"2,184.91"
France,"184,493.00",382,87,"2,120.61"
Australia,"138,103.81",56,9,"15,344.87"
Spain,"55,706.56",88,30,"1,856.89"
Switzerland,"53,065.60",50,21,"2,526.93"


**So-what.** The UK is **85% of revenue**. Internationally revenue per customer is *wildly* higher: the **Netherlands (9 customers → £284k)** and **EIRE (3 customers → £271k)** are a handful of B2B accounts, not consumer markets. Germany and France are the only international markets with real customer breadth (~90 customers each). Losing one Dutch account would hurt more than losing 100 UK consumers.

## Baskets · is there wholesale behaviour?

In [12]:
viz.basket_hist(orders)

In [13]:
print(orders[["Revenue","Units","Lines"]].describe(percentiles=[.5,.9,.99]).round(0))
big = orders.Revenue >= 500
print(f"Orders >= £500: {big.mean():.1%} of orders but {orders[big].Revenue.sum()/orders.Revenue.sum():.1%} of revenue")
print(f"Orders with >= 100 units: {(orders.Units>=100).mean():.1%}")

         Revenue     Units     Lines
count  19,773.00 19,773.00 19,773.00
mean      518.00    281.00     26.00
std     1,764.00    959.00     47.00
min         0.00      1.00      1.00
50%       302.00    152.00     15.00
90%       916.00    531.00     52.00
99%     4,409.00  2,233.00    218.00
max   168,470.00 80,995.00  1,109.00
Orders >= £500: 23.8% of orders but 66.3% of revenue
Orders with >= 100 units: 65.3%


**So-what.** The median order is **152 units / £302** — far beyond a household shopping basket — and 65% of orders contain 100+ units. Mean (£518) ≫ median (£302): a long wholesale tail. **24% of orders (≥£500) generate 66% of revenue.** This is a trade business wearing a consumer storefront, so segmentation must separate bulk buyers from small-batch buyers (guideline: protect the core revenue line).

## Returns · how common, and who?

In [14]:
viz.returns_over_time(df)

In [15]:
viz.returns_by_product(df)

In [16]:
canc_inv = df[df.IsCancellation].InvoiceNo.nunique() / df.InvoiceNo.nunique()
net_prod = df[df.IsCancellation & ~df.IsNonProduct & (df.UnitPrice>0)].TotalLineRevenue.sum()
print(f"Cancellation invoices: {canc_inv:.1%} of all invoices | returned value = {-net_prod/sales.TotalLineRevenue.sum():.1%} of gross product sales")
rc = df[df.IsCancellation & df.CustomerID.notna()].groupby("CustomerID").TotalLineRevenue.sum().nsmallest(5)
print("Customers with the largest return value:"); rc

Cancellation invoices: 14.8% of all invoices | returned value = 4.6% of gross product sales
Customers with the largest return value:


CustomerID
16,446.00   -168,469.60
12,346.00    -77,183.60
15,098.00    -39,267.00
16,029.00    -27,682.15
15,749.00    -22,998.40
Name: TotalLineRevenue, dtype: float64

**So-what.** Roughly **15% of invoices are cancellations but they only equal ~4.6% of gross sales value** — many small corrections, a few huge reversals. Returns are concentrated: customer 16446 alone cancelled £168k (one order), and a few B2B accounts account for most of the value. Return policy design should focus on a short list of accounts and SKUs rather than the whole base.

## Retention · are customers coming back?

In [17]:
viz.cohort_heatmap(df)

**So-what.** The *December 2010* cohort (the oldest and largest, 884 customers) keeps roughly **32–40%** of its customers buying in every later month — clearly stickier than the 2011 cohorts, which settle at ~**17–30%**. Newer cohorts either reflect weaker acquisition quality or are simply too young; the sample cannot say which (descriptive data, no causal claim).

## Business insights (10+)
1. **Trade, not consumer.** Median order = 152 units; 65% of orders ≥100 units → treat the base as B2B-first.
2. **Revenue is front-loaded to Q4.** Nov ≈ 2.2× the Mar–Aug average; the ramp begins in September.
3. **A 10:00–15:00 weekday business.** No Saturday trading; Thursday strongest, Sunday weakest.
4. **Product Pareto holds:** ~21.5% of SKUs = 80% of net revenue; ~3,000 SKUs are long-tail.
5. **Gross rankings lie.** Two of the top-3 unit "bestsellers" were fully cancelled; only net-of-returns ranking is trustworthy.
6. **Top 20% of customers → ~74% of revenue; top 10 customers alone → ~17%.** Dependency risk.
7. **UK = 85% of product revenue**, but international is a few big B2B accounts (NL, EIRE) → high value, high concentration risk.
8. **Guest checkout is 14.7% of product revenue** (3,710 invoices, always fully anonymous) — real money that no CRM can retarget today.
9. **Returns:** ~15% of invoices, ~4.6% of value, highly concentrated in few accounts.
10. **22 former volume SKUs sold nothing in the last 90 days** → check for unintended stock-outs.
11. **65% of identified customers bought more than once** — but half of all customers last ordered 51+ days before the snapshot, a churn-watch signal.
12. **The oldest cohort is the stickiest** — loyalty value is created in the first months.